In [35]:
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import DoclingDocument
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions

ROOT = Path.cwd().parent                          # notebook lives in notebooks/
PDF = ROOT / "data" / "01_09_blueprints-visual-scripting_LQ.pdf"
PARSED = ROOT / "data" / "parsed"
PAGE_RANGE = (1, 580)
DOC_JSON = PARSED / f"blueprints_p{PAGE_RANGE[0]}-{PAGE_RANGE[1]}.json"


def convert_pdf() -> DoclingDocument:
    options = PdfPipelineOptions()
    options.do_ocr = False
    options.generate_picture_images = False
    options.accelerator_options = AcceleratorOptions(device=AcceleratorDevice.CUDA)
    converter = DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=options)}
    )
    return converter.convert(PDF).document


def load_document() -> DoclingDocument:
    """Load the cached parse if it exists; otherwise convert once and cache it."""
    if DOC_JSON.exists():
        return DoclingDocument.load_from_json(DOC_JSON)
    doc = convert_pdf()
    PARSED.mkdir(parents=True, exist_ok=True)
    doc.save_as_json(DOC_JSON)
    return doc




In [36]:
import torch
print(torch.__version__, torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.14.1+cu126 True
NVIDIA GeForce RTX 3060 Laptop GPU


In [37]:
import sys, torch
print(sys.executable)
print(torch.__version__, torch.version.cuda, torch.cuda.is_available())

c:\Users\Ayah\Documents\VSCODE\UE5.7-RAG\.venv\Scripts\python.exe
2.14.1+cu126 12.6 True


In [38]:
doc = load_document()
print(DOC_JSON.name, "| tables:", len(doc.tables))

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

blueprints_p1-580.json | tables: 99


In [39]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

table = doc.tables[-4]
table_df = table.export_to_dataframe()
table_df


Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


,Button,Description
0,,Compilation was successful. Clicking the button compiles the Blueprint being edited. Output from the compiling process is displayed in the Blueprint Log of the Message Log. This button will be inactive during debugging.


In [40]:
for table in doc.tables:
    print(table.export_to_dataframe(doc=doc).shape)

(6, 2)
(4, 3)
(1, 2)
(3, 2)
(5, 2)
(5, 2)
(2, 2)
(3, 2)
(1, 2)
(2, 1)
(2, 1)
(9, 2)
(5, 2)
(10, 2)
(7, 2)
(2, 2)
(2, 2)
(3, 2)
(3, 2)
(1, 2)
(2, 2)
(4, 2)
(2, 2)
(2, 2)
(4, 2)
(7, 2)
(3, 1)
(8, 1)
(4, 3)
(4, 3)
(4, 3)
(4, 3)
(4, 3)
(4, 3)
(2, 3)
(2, 3)
(2, 3)
(2, 3)
(3, 3)
(3, 3)
(3, 3)
(4, 3)
(2, 3)
(2, 3)
(3, 3)
(2, 3)
(3, 2)
(2, 2)
(3, 2)
(12, 3)
(6, 2)
(6, 2)
(1, 2)
(3, 2)
(1, 1)
(6, 2)
(6, 2)
(1, 2)
(7, 2)
(6, 2)
(9, 2)
(5, 2)
(6, 2)
(10, 2)
(6, 2)
(6, 2)
(5, 2)
(6, 2)
(8, 2)
(5, 2)
(4, 1)
(8, 2)
(2, 2)
(8, 2)
(5, 2)
(6, 2)
(5, 2)
(3, 2)
(3, 2)
(3, 2)
(4, 3)
(5, 2)
(1, 2)
(5, 2)
(1, 2)
(12, 2)
(12, 2)
(12, 2)
(3, 2)
(5, 2)
(2, 2)
(6, 2)
(8, 2)
(8, 2)
(5, 3)
(1, 2)
(13, 2)
(5, 2)
(4, 2)


In [41]:
import re
def clean(text) -> str:
    text = str(text).replace("Downloaded from UnrealDocs.net", "")
    return re.sub(r"\s+", " ", text).strip()


def is_headerless(df) -> bool:
    """Docling names columns 0, 1, 2... when it found no header row."""
    return [str(c) for c in df.columns] == [str(i) for i in range(df.shape[1])]


def has_duplicate_header(df) -> bool:
    """First data row repeats the column names."""
    if len(df) == 0:
        return False
    return [clean(v) for v in df.iloc[0]] == [clean(c) for c in df.columns]


def same_header(df, ref) -> bool:
    """Header printed again on the new page: identical column names."""
    return [clean(c) for c in df.columns] == [clean(c) for c in ref.columns]

In [42]:
from docling_core.types.doc import DocItemLabel, TableItem

FURNITURE = {DocItemLabel.PAGE_HEADER, DocItemLabel.PAGE_FOOTER}

def content_before_tables(doc):
    """Map each table's ref -> True if real content sits between it and the previous table."""
    gap_before, gap = {}, True
    for item, _ in doc.iterate_items():
        if isinstance(item, TableItem):
            gap_before[item.self_ref] = gap
            gap = False
        elif getattr(item, "label", None) not in FURNITURE:
            gap = True                      # a title, paragraph, picture... anything real
    return gap_before

In [43]:
def stitch_tables(doc):
    gap_before = content_before_tables(doc)
    tables = []
    for t in doc.tables:
        if not t.prov:
            continue
        page, df = t.prov[0].page_no, t.export_to_dataframe(doc)

        if tables:
            prev = tables[-1]
            if (not gap_before[t.self_ref]                       # <-- new: nothing between them
                    and page == prev["pages"][-1] + 1
                    and df.shape[1] == prev["df"].shape[1]
                    and (is_headerless(df) or same_header(df, prev["df"]) or has_duplicate_header(df))):
                if has_duplicate_header(df):
                    df = df.iloc[1:]
                df = df.set_axis(prev["df"].columns, axis=1)
                prev["df"] = pd.concat([prev["df"], df], ignore_index=True)
                prev["pages"].append(page)
                continue

        tables.append({"df": df, "pages": [page]})
    return tables

In [44]:
tables = stitch_tables(doc)
print(len(doc.tables), "fragments ->", len(tables), "tables")
for t in tables:
        print(t["pages"], t["df"].shape, list(t["df"].columns)[:3])

99 fragments -> 89 tables
[136] (6, 2) ['Action', 'Command']
[184] (4, 3) ['Filter', 'Searchable Tags', 'Subfilters']
[185] (1, 2) ['Filter Searchable Tags', 'Subfilters']
[198] (3, 2) ['Keyword', 'Description Exposes this class as an acceptable base class for']
[199] (5, 2) ['Keyword', 'Description']
[201, 202] (7, 2) ['Keyword', 'Description']
[218, 219] (4, 2) ['Item', 'Description']
[257] (2, 1) ['Item Description']
[258] (2, 1) ['Item Description']
[259] (9, 2) ['Item', 'Description']
[260] (5, 2) ['Item', 'Description']
[261] (10, 2) ['Item', 'Description']
[262] (7, 2) ['Item', 'Description']
[265] (2, 2) ['Item', 'Description']
[266] (2, 2) ['Item', 'Description']
[267] (3, 2) ['Item', 'Description']
[277] (3, 2) ['Setting', 'Description']
[308] (1, 2) ['Node', 'Description']
[309] (2, 2) ['Node', 'Description']
[310, 311] (6, 2) ['Node', 'Description']
[311] (2, 2) ['Node', 'Description']
[312] (4, 2) ['Node', 'Description']
[315] (7, 2) ['Function', 'Description']
[320, 321] 

## Table context: heading path + preceding text

Keyed by the table fragment's `self_ref`, so a stitched table keeps the context of
its first fragment.

In [45]:
from docling_core.types.doc import DocItemLabel, TableItem

TEXT_LABELS = {
    DocItemLabel.TEXT, DocItemLabel.PARAGRAPH, DocItemLabel.LIST_ITEM,
    DocItemLabel.CAPTION, DocItemLabel.FOOTNOTE, DocItemLabel.CODE,
}
MAX_ABOVE_TEXTS = 3
MAX_ABOVE_CHARS = 600


def squash(text: str) -> str:
    """Collapse whitespace so context stays compact in the prompt."""
    return re.sub(r"\s+", " ", text).strip()


def table_contexts(doc):
    """Map each table fragment ref -> {"headings", "above"} in reading order."""
    contexts, heading_stack, buffer = {}, [], []
    for item, _ in doc.iterate_items():
        label = getattr(item, "label", None)
        if label in FURNITURE:
            continue
        if isinstance(item, TableItem):
            contexts[item.self_ref] = {
                "headings": " > ".join(text for _, text in heading_stack),
                "above": " ".join(buffer[-MAX_ABOVE_TEXTS:])[-MAX_ABOVE_CHARS:],
            }
            buffer = []
        elif label == DocItemLabel.SECTION_HEADER:
            while heading_stack and heading_stack[-1][0] >= item.level:
                heading_stack.pop()
            heading_stack.append((item.level, squash(item.text)))
            buffer = []
        elif label in TEXT_LABELS:
            buffer.append(squash(item.text))
    return contexts

## Table -> JSON

Headerless tables (columns literally named 0, 1, 2...) export `columns: None`.

In [46]:
def cell_text(value) -> str:
    """DataFrame cell -> string; missing values become empty strings."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return str(value).strip()


def table_json(df) -> dict:
    """Structured copy of a table: {"columns": [...] | None, "rows": [[...], ...]}."""
    columns = None if is_headerless(df) else [str(c) for c in df.columns]
    return {"columns": columns,
            "rows": [[cell_text(v) for v in row] for row in df.itertuples(index=False)]}


def rows_as_text(table: dict) -> str:
    """Plain-text rendering of the rows, used for embed and return text."""
    lines = []
    for row in table["rows"]:
        if table["columns"] is None:
            lines.append(" | ".join(row))
        else:
            lines.append("; ".join(f"{name}: {value}" for name, value in zip(table["columns"], row)))
    return "\n".join(lines)

## Local-LLM summary (Ollama)

Gemma has no reliable system role, so everything goes in one user message.
Summaries are cached by prompt hash: each table is only run once.

In [47]:
import hashlib
import json

from pathlib import Path



import ollama



MODEL = "gemma2:2b"

MAX_TABLE_CHARS = 6000

MIN_ROWS_TO_SUMMARIZE = 3

SUMMARY_CACHE = Path("..") / "data" / "parsed" / "table_summaries.json"

OLLAMA_OPTIONS = {"temperature": 0, "num_predict": 256}



INSTRUCTIONS = (

    "You are summarizing one table from the Unreal Engine documentation. "

    "Write 2-3 sentences on what the table documents and what a reader could look "

    "up in it. Copy node, pin and checkbox names exactly as written. Use only the "

    "table and the context below; add nothing else. No preamble, no bullet points."

)





def load_summaries() -> dict:

    if SUMMARY_CACHE.exists():

        return json.loads(SUMMARY_CACHE.read_text(encoding="utf-8"))

    return {}





def save_summaries(cache: dict) -> None:

    SUMMARY_CACHE.parent.mkdir(parents=True, exist_ok=True)

    SUMMARY_CACHE.write_text(json.dumps(cache, indent=2, ensure_ascii=False), encoding="utf-8")





def prompt_key(prompt: str) -> str:

    return hashlib.sha256(prompt.encode("utf-8")).hexdigest()





def fit_table(table: dict, limit: int):

    """Keep only the leading rows that fit the context budget; report truncation."""

    if len(json.dumps(table, ensure_ascii=False)) <= limit:

        return table, False

    rows = []

    for row in table["rows"]:

        if len(json.dumps({"columns": table["columns"], "rows": rows + [row]}, ensure_ascii=False)) > limit:

            break

        rows.append(row)

    return {"columns": table["columns"], "rows": rows}, True





def build_prompt(table: dict, headings: str, above: str) -> str:

    return (

        f"{INSTRUCTIONS}\n\n"

        f"Section: {headings or '(none)'}\n"

        f"Text above the table: {above or '(none)'}\n\n"

        f"Table JSON:\n{json.dumps(table, ensure_ascii=False)}"

    )





def summarize_table(table: dict, headings: str, above: str, cache: dict) -> dict:

    """Return {"summary", "summary_truncated"}, memoized in cache by prompt hash."""

    if not table["rows"] or len(table["rows"]) <= MIN_ROWS_TO_SUMMARIZE:

        return {"summary": None, "summary_truncated": False}



    fitted, truncated = fit_table(table, MAX_TABLE_CHARS)

    prompt = build_prompt(fitted, headings, above)

    key = prompt_key(prompt)

    if key not in cache:

        response = ollama.chat(

            model=MODEL,

            messages=[{"role": "user", "content": prompt}],

            options=OLLAMA_OPTIONS,

        )

        cache[key] = response["message"]["content"].strip()

        save_summaries(cache)

    return {"summary": cache[key], "summary_truncated": truncated}

## Build records

In [48]:
def table_record(df, pages, headings, above, summary) -> dict:
    """One retrievable record per table."""
    table = table_json(df)
    raw_rows = rows_as_text(table)
    parts = [part for part in (headings, above, summary["summary"], raw_rows) if part]
    without_summary = [part for part in (headings, above, raw_rows) if part]
    return {
        "pages": list(pages),
        "headings": headings,
        "above": above,
        "summary": summary["summary"],
        "summary_truncated": summary["summary_truncated"],
        "table": table,
        "embed_text": "\n\n".join(parts),
        "return_text": "\n\n".join(without_summary),
    }


def attach_context(tables, contexts, cache) -> list:
    """Walk stitched tables in order; each consumes as many fragments as it spans pages."""
    records, fragment = [], 0
    for entry in tables:
        context = contexts.get(doc.tables[fragment].self_ref, {"headings": "", "above": ""})
        summary = summarize_table(table_json(entry["df"]), context["headings"], context["above"], cache)
        records.append(table_record(entry["df"], entry["pages"],
                                    context["headings"], context["above"], summary))
        fragment += len(entry["pages"])
    return records

In [49]:
contexts = table_contexts(doc)
cache = load_summaries()
records = attach_context(tables, contexts, cache)
print(len(records), "records;",
      sum(1 for r in records if r["summary"]), "summarized;",
      sum(1 for r in records if r["summary_truncated"]), "truncated")

89 records; 57 summarized; 0 truncated


## Verify

One merged table, one headerless, one skipped-small, one truncated (if any), one regular.

In [50]:
def pick(predicate):
    """First record matching predicate that has not been shown yet."""
    return next((r for r in records if id(r) not in seen and predicate(r)), None)


seen = set()
samples = [
    ("merged across pages", lambda r: len(r["pages"]) > 1),
    ("headerless", lambda r: r["table"]["columns"] is None),
    ("skipped small", lambda r: r["summary"] is None),
    ("truncated", lambda r: r["summary_truncated"]),
    ("regular", lambda r: r["table"]["columns"] is not None
     and len(r["table"]["rows"]) > MIN_ROWS_TO_SUMMARIZE),
]

for label, predicate in samples:
    record = pick(predicate)
    if record is None:
        print(f"--- {label}: none in this range ---")
        continue
    seen.add(id(record))
    print(f"--- {label} ---")
    print(json.dumps(record, indent=2, ensure_ascii=False)[:1500])

--- merged across pages ---
{
  "pages": [
    201,
    202
  ],
  "headings": "Executable and Overridable Functions",
  "above": "UPARAM() to change the display name of a pin. For example, the MakeRotator function in KismetMathLibrary uses UPARAM() and the DisplayName keyword to change how the Roll, Pitch, and Yaw parameters appear in Blueprints. /** Makes a rotator {Roll, Pitch, Yaw} from rotation values supplied in degrees */ UFUNCTION(BlueprintPure, Category=\"Math|Rotator\" , meta= (Keywords=\"construct build rotation rotate rotator makerotator\" NativeMakeFunc)) static FRotator MakeRotator( UPARAM(DisplayName=\"X (Roll)\") float Roll, UPARAM(DisplayName=\"Y (Pitch)\") float Pitch, UPARAM(DisplayName=\"Z (Yaw)\") float Yaw); ,",
  "summary": "This table describes different types of functions that can be used in Unreal Engine Blueprints, including how they communicate with native code.  Readers can look up specific keywords like \"BlueprintPure\" and \"BlueprintCallable\" to unders